# FlashVSR+ Video Upscaler (Kaggle / Colab)

One-step diffusion video super-resolution: 4x upscale of low-res videos with temporal consistency.
Models: Turbo (fast) / Base (best detail). Requires **GPU T4 x2** (Kaggle) or any 16GB+ GPU.

- Official repo: https://github.com/OpenImagingLab/FlashVSR_Plus
- Weights: https://huggingface.co/ruchangyao/FlashVSR_plus
- Output sizes must be multiples of 128. 1080p -> request 1920x1152; 1440p -> 2560x1408.

In [ ]:
# Step 1: Setup
import os, subprocess, sys, time

REPO = "FlashVSR_Plus"
COMMIT = "b548a2cc2c929d3d4676f2973abe27876ec4dee7"  # pinned 16 Oct 2026

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "init", "-q", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "add", "origin",
                    "https://github.com/OpenImagingLab/FlashVSR_Plus.git"], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "FETCH_HEAD"], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-e", ".", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "safetensors", "einops", "imageio-ffmpeg", "gradio"], check=True)
print("setup OK")

setup OK


In [ ]:
# Step 2: Download Turbo weights (~7 GB)
# turbo: dit.safetensors (5.7GB) + lq_proj_in.safetensors (1.2GB) + shared TCDecoder.ckpt
import os
from huggingface_hub import hf_hub_download

REPO_ID = "ruchangyao/FlashVSR_plus"
REV = "46b79520946ac67dec94d1566490690953c7debc"  # pinned 14 Oct 2026

os.makedirs("ckpts/turbo", exist_ok=True)
for f in ["turbo/dit.safetensors", "turbo/lq_proj_in.safetensors"]:
    p = hf_hub_download(repo_id=REPO_ID, filename=f, revision=REV)
    dst = os.path.join("ckpts", os.path.basename(os.path.dirname(p)), os.path.basename(p))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        os.replace(p, dst)

p = hf_hub_download(repo_id=REPO_ID, filename="TCDecoder.ckpt", revision=REV)
if not os.path.exists("ckpts/TCDecoder.ckpt"):
    os.replace(p, "ckpts/TCDecoder.ckpt")
print("weights OK")
print(os.listdir("ckpts"), os.listdir("ckpts/turbo"))

weights OK
['turbo', '.gitkeep', 'TCDecoder.ckpt'] ['lq_proj_in.safetensors', 'dit.safetensors']


In [ ]:
# Step 3 & 4: Interactive Web UI for Upscaling
import gradio as gr
import subprocess
import os
import sys

def upscale_video(input_video, mode, resolution):
    if not input_video:
        return None, "कृपया पहले एक वीडियो अपलोड करें!"

    # Map chosen resolution to dimensions divisible by 128
    res_map = {
        "720p (1280x768)": (1280, 768),
        "1080p (1920x1152)": (1920, 1152),
        "1440p (2560x1408)": (2560, 1408)
    }

    W, H = res_map.get(resolution, (1280, 768))
    output_path = "/kaggle/working/output.mp4"

    # Ensure working directory is correct
    if os.path.basename(os.getcwd()) != "FlashVSR_Plus":
        if os.path.exists("FlashVSR_Plus"):
            os.chdir("FlashVSR_Plus")

    cmd = [
        sys.executable, "-m", "inference.run_stream",
        "--input", input_video,
        "--mode", mode,
        "--output", output_path,
        "--width", str(W), "--height", str(H),
        "--decode-every", "28",
        "--hardware-profile", "generic"
    ]

    print("Running Command:", " ".join(cmd))

    try:
        subprocess.run(cmd, check=True)
        return output_path, "Upscaling Completed successfully!"
    except Exception as e:
        return None, f"Error occurred: {str(e)}"

# Design custom theme and interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🎬 FlashVSR+ Video Upscaler UI")
    gr.Markdown("Upload your video, choose target resolution, and click upscale to run!")

    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="Input Video")
            mode_input = gr.Dropdown(choices=["turbo", "base"], value="turbo", label="Model Mode")
            res_input = gr.Dropdown(
                choices=["720p (1280x768)", "1080p (1920x1152)", "1440p (2560x1408)"],
                value="720p (1280x768)",
                label="Target Resolution"
            )
            btn = gr.Button("🚀 Start Upscaling", variant="primary")

        with gr.Column():
            video_output = gr.Video(label="Upscaled Output Video")
            status_output = gr.Textbox(label="Status Logs", interactive=False)

    btn.click(
        fn=upscale_video,
        inputs=[video_input, mode_input, res_input],
        outputs=[video_output, status_output]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_7951/1430854681.py:45: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64eb37724edcd016d7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Running Command: /usr/bin/python3 -m inference.run_stream --input /tmp/gradio/375e76c884b321f6dad50f2acf84382e6426f10d1c34187a23b8f9f4af2f2dfd/Man_leaning_against_yellow_car_0c915f85.mp4 --mode turbo --output /kaggle/working/output.mp4 --width 1280 --height 768 --decode-every 28 --hardware-profile generic
Running Command: /usr/bin/python3 -m inference.run_stream --input /tmp/gradio/375e76c884b321f6dad50f2acf84382e6426f10d1c34187a23b8f9f4af2f2dfd/Man_leaning_against_yellow_car_0c915f85.mp4 --mode turbo --output /kaggle/working/output.mp4 --width 1280 --height 768 --decode-every 28 --hardware-profile generic
Running Command: /usr/bin/python3 -m inference.run_stream --input /tmp/gradio/375e76c884b321f6dad50f2acf84382e6426f10d1c34187a23b8f9f4af2f2dfd/Man_leaning_against_yellow_car_0c915f85.mp4 --mode turbo --output /kaggle/working/output.mp4 --width 1280 --height 768 --decode-every 28 --hardware-profile generic
Running Command: /usr/bin/python3 -m inference.run_stream --input /tmp/gradio/3

In [ ]:
# Step 4: Run upscale
# Pick output size (multiples of 128). Turbo mode = fast.
OUT_W = 1920   # e.g. 1280 (720p), 1920 (1080p), 2560 (1440p)
OUT_H = 1152   # must be divisible by 128
MODE = "turbo"  # or "base"

cmd = [sys.executable, "-m", "inference.run_stream",
       "--input", INPUT,
       "--mode", MODE,
       "--output", "/kaggle/working/output.mp4",
       "--width", str(OUT_W), "--height", str(OUT_H),
       "--decode-every", "28",
       "--hardware-profile", "generic"]   # SDPA fallback for T4

print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("DONE -> /kaggle/working/output.mp4")

NameError: name 'INPUT' is not defined

In [ ]:
# Download result
from IPython.display import Video
Video("/kaggle/working/output.mp4", width=640, embed=True)